# Personal Game — Kaggle node

Thin wrapper around `kaggle/runner.py` — all logic lives there.

**Before running:**
1. Set `REPO_URL` below (your fork of personal-game).
2. Add Kaggle **Secrets** (Add-ons → Secrets): `NODE_ENROLLMENT_TOKEN`,
   `PG_API_TOKEN` (if the control plane requires one). Never hardcode them.
3. This environment advertises `STREAMING_ALLOWED=false` by default:
   Kaggle's AUP forbids game streaming, so this node will never receive
   game sessions. It is for diagnostics/lifecycle/capability testing.
4. The control plane used by default is the Personal Game Cloudflare edge
   relay: `https://personal-game-relay.rahul-zed-relay-84739261.workers.dev/personal-game`
   (control HTTP only — Moonlight media never goes through it).


In [ ]:
# CELL 1 — Clone the repository + load Kaggle Secrets into env
from kaggle_secrets import UserSecretsClient
import os

REPO_URL = "https://github.com/YOUR-USER/personal-game.git"  # <-- set this
secrets = UserSecretsClient()
for key in ("NODE_ENROLLMENT_TOKEN", "PG_API_TOKEN", "CONTROL_PLANE_URL",
            "NODE_NAME", "STREAMING_ALLOWED", "MEDIA_NETWORK", "MEDIA_ENDPOINT"):
    try:
        val = secrets.get_secret(key)
        if val:
            os.environ[key] = val
    except Exception:
        pass  # secret not defined — the runner reports what is missing

!python kaggle/runner.py clone --repo-url "$REPO_URL" 2>/dev/null || python personal-game/github/kaggle/runner.py clone --repo-url "$REPO_URL"

In [ ]:
# CELL 2 — Diagnostics (honest capability report + control-plane reachability)
!python personal-game/github/kaggle/runner.py diagnostics

In [ ]:
# CELL 3 — Setup (Go toolchain check + agent build, idempotent)
!python personal-game/github/kaggle/runner.py setup

In [ ]:
# CELL 4 — Run the shared node agent (foreground; interrupt the cell to stop)
!python personal-game/github/kaggle/runner.py run

In [ ]:
# CELL 5 — Cleanup (ephemeral staging only, never saves)
!python personal-game/github/kaggle/runner.py cleanup